# EEG Conformer — Intra-Subject

Intra-subject MEG brain-state classification with **EEGConformerGAP**:
ConvModule (temporal + spatial CNN) + positional embedding + Transformer encoder + GAP.

Pipeline:
1. Load and preprocess data via shared `protocol_utils`
2. Run diagnostic training on train/val split to find best epoch via val loss
3. Retrain on all training files for that epoch count
4. Evaluate once on held-out test set

## 1. Imports

In [ ]:
import sys, math
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader

sys.path.append('..')
from protocol_utils import (
    DatasetPaths, PreprocessConfig,
    list_files, make_windows, split_files_train_val,
    summarize_logits, CLASS_NAMES,
    file_accuracy_from_logits,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 2. Data Loading

Protocol-fixed file-level split: `val_frac=0.2, seed=0`.
Preprocessing: downsample ×4 (~508 Hz), z-score per sensor, windows of 512 samples (~1 s), stride 128.

In [ ]:
paths = DatasetPaths(data_root='../Final Project data')
cfg   = PreprocessConfig(downsample_factor=4, window_size=512, stride=128)

train_files, val_files = split_files_train_val(
    list_files(paths.intra_train), val_frac=0.2, seed=0
)
test_files = list_files(paths.intra_test)

X_train, y_train, _           = make_windows(train_files, cfg)
X_val,   y_val,   val_groups  = make_windows(val_files,   cfg)
X_test,  y_test,  test_groups = make_windows(test_files,  cfg)

print(f'Train : {X_train.shape}  ({len(train_files)} files)')
print(f'Val   : {X_val.shape}  ({len(val_files)} files)')
print(f'Test  : {X_test.shape}  ({len(test_files)} files)')

## 3. Model

In [ ]:
class ConvModule(nn.Module):
    def __init__(self, n_channels=248, n_filters=40, kernel_temp=25,
                 pool_size=75, pool_stride=15, dropout=0.5):
        super().__init__()
        self.temporal = nn.Sequential(
            nn.Conv2d(1, n_filters, kernel_size=(1, kernel_temp), bias=False),
            nn.BatchNorm2d(n_filters),
        )
        self.spatial = nn.Sequential(
            nn.Conv2d(n_filters, n_filters, kernel_size=(n_channels, 1), bias=False),
            nn.BatchNorm2d(n_filters),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, pool_size), stride=(1, pool_stride)),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        x = self.temporal(x)
        x = self.spatial(x)
        x = x.squeeze(2)
        return x.permute(0, 2, 1)  # (B, T_seq, n_filters)


class EEGConformerGAP(nn.Module):
    """ConvModule + positional embedding + TransformerEncoder + GAP + linear."""
    def __init__(self, n_channels=248, n_classes=4, window_size=512,
                 n_filters=40, kernel_temp=25, pool_size=75, pool_stride=15,
                 num_heads=8, num_layers=2, ff_dim=80, dropout=0.5):
        super().__init__()
        self.conv = ConvModule(n_channels, n_filters, kernel_temp,
                               pool_size, pool_stride, dropout)
        T_prime = window_size - kernel_temp + 1
        T_seq   = (T_prime - pool_size) // pool_stride + 1
        self.pos_embedding = nn.Parameter(torch.randn(1, T_seq, n_filters) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=n_filters, nhead=num_heads,
            dim_feedforward=ff_dim, dropout=dropout,
            activation='gelu', batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm        = nn.LayerNorm(n_filters)
        self.classifier  = nn.Linear(n_filters, n_classes)

    def forward(self, x):
        x = self.conv(x)               # (B, T_seq, n_filters)
        x = x + self.pos_embedding
        x = self.transformer(x)
        x = self.norm(x)
        x = x.mean(dim=1)              # GAP
        return self.classifier(x)


# kernel_temp=51 -> T' = 462, T_seq = (462-75)//15 + 1 = 26 tokens
_m  = EEGConformerGAP(kernel_temp=51, num_layers=2).to(device)
_in = torch.zeros(2, 1, 248, 512).to(device)
print(f'EEGConformerGAP parameters: {sum(p.numel() for p in _m.parameters()):,}')
print(f'Output shape: {tuple(_m(_in).shape)}')
del _m, _in

## 4. Training Utilities

In [ ]:
class MEGWindowDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return (torch.from_numpy(self.X[idx]).unsqueeze(0),
                torch.tensor(self.y[idx], dtype=torch.long))


def run_epoch(loader, model, criterion, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss, correct, n = 0.0, 0, 0
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for x, y in loader:
            x, y   = x.to(device), y.to(device)
            logits = model(x)
            loss   = criterion(logits, y)
            if training:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * len(y)
            correct    += (logits.argmax(1) == y).sum().item()
            n          += len(y)
    return total_loss / n, correct / n


def get_val_file_acc(model, X_val, y_val, val_groups, batch_size=128):
    model.eval()
    all_logits = []
    with torch.no_grad():
        for i in range(0, len(X_val), batch_size):
            x = torch.from_numpy(X_val[i:i+batch_size]).unsqueeze(1).to(device)
            all_logits.append(model(x).cpu().numpy())
    return file_accuracy_from_logits(np.concatenate(all_logits, axis=0), y_val, val_groups)

## 5. Hyperparameters and Data Loaders

In [ ]:
# Best config from grid search (fixed from CNN-only intra search)
best = {'kernel_temp': 51, 'n_filters': 40, 'dropout': 0.3}

train_loader = DataLoader(MEGWindowDataset(X_train, y_train),
                          batch_size=64, shuffle=True, num_workers=0)

all_train_files = train_files + val_files
X_all, y_all, _ = make_windows(all_train_files, cfg)
all_loader = DataLoader(MEGWindowDataset(X_all, y_all),
                        batch_size=64, shuffle=True, num_workers=0)
print(f'Full training set: {X_all.shape}  ({len(all_train_files)} files)')

## 6. Diagnostic — Overfitting Check

Train on the protocol train split (26 files) with both train and val loss tracked each epoch.
Checkpoints on **val loss** (continuous signal, avoids coarse file-acc steps).
The best epoch `_best_epoch` is used directly as the final training epoch count.

In [ ]:
_diag_model     = EEGConformerGAP(
    n_channels=248, n_classes=4, window_size=512,
    n_filters=best['n_filters'], kernel_temp=best['kernel_temp'],
    dropout=best['dropout'], num_layers=2,
).to(device)
_diag_optimizer = torch.optim.Adam(_diag_model.parameters(), lr=5e-4, weight_decay=1e-4)
_diag_criterion = nn.CrossEntropyLoss()
_val_loader     = DataLoader(MEGWindowDataset(X_val, y_val),
                             batch_size=128, shuffle=False, num_workers=0)

_MAX_EPOCHS, _PATIENCE = 40, 10
_diag_history = {'train_loss': [], 'val_loss': [], 'val_file_acc': []}
_best_val_loss, _best_epoch, _patience_counter = float('inf'), 0, 0

print(f'Diagnostic — kernel_temp={best["kernel_temp"]}, '
      f'n_filters={best["n_filters"]}, dropout={best["dropout"]}\n')

for _epoch in range(1, _MAX_EPOCHS + 1):
    _train_loss, _ = run_epoch(train_loader, _diag_model, _diag_criterion, _diag_optimizer)
    _val_loss,   _ = run_epoch(_val_loader,  _diag_model, _diag_criterion)
    _val_file_acc  = get_val_file_acc(_diag_model, X_val, y_val, val_groups)
    _diag_history['train_loss'].append(_train_loss)
    _diag_history['val_loss'].append(_val_loss)
    _diag_history['val_file_acc'].append(_val_file_acc)
    if _val_loss < _best_val_loss:
        _best_val_loss, _best_epoch, _patience_counter = _val_loss, _epoch, 0
    else:
        _patience_counter += 1
        if _patience_counter >= _PATIENCE:
            print(f'  Early stop at epoch {_epoch}')
            break
    marker = '  *' if _epoch == _best_epoch else ''
    print(f'  ep {_epoch:02d}  train {_train_loss:.4f}  val {_val_loss:.4f}'
          f'  file_acc {_val_file_acc:.3f}{marker}')

_ep_range = range(1, len(_diag_history['train_loss']) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(_ep_range, _diag_history['train_loss'], color='steelblue',  label='Train loss')
ax1.plot(_ep_range, _diag_history['val_loss'],   color='darkorange', label='Val loss')
ax1.axvline(_best_epoch, color='red', linestyle='--', linewidth=1.2,
            label=f'Best epoch ({_best_epoch})')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('EEGConformerGAP — Train vs Val Loss'); ax1.legend()

ax2.plot(_ep_range, _diag_history['val_file_acc'], color='seagreen', marker='o', markersize=3)
ax2.axvline(_best_epoch, color='red', linestyle='--', linewidth=1.2,
            label=f'Best epoch ({_best_epoch})')
ax2.axhline(0.25, color='grey', linestyle=':', linewidth=0.8, label='Chance (25%)')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Val file accuracy'); ax2.set_ylim(0, 1); ax2.legend()
plt.suptitle(f'Diagnostic — kernel_temp={best["kernel_temp"]}, n_filters={best["n_filters"]}, '
             f'dropout={best["dropout"]}, num_layers=2', fontsize=10)
plt.tight_layout(); plt.show()

print(f'\nBest val loss at epoch {_best_epoch}  '
      f'(val_loss={_best_val_loss:.4f}, '
      f'val_file_acc={_diag_history["val_file_acc"][_best_epoch-1]:.3f})')
del _diag_model, _diag_optimizer, _diag_criterion, _val_loader

## 7. Final Training

In [ ]:
conformer_cv_best_epoch = _best_epoch
print(f'Using epoch {conformer_cv_best_epoch} from diagnostic run (val-loss minimum)')

In [ ]:
final_conformer = EEGConformerGAP(
    n_channels=248, n_classes=4, window_size=512,
    n_filters=best['n_filters'], kernel_temp=best['kernel_temp'],
    dropout=best['dropout'], num_layers=2,
).to(device)
optimizer = torch.optim.Adam(final_conformer.parameters(), lr=5e-4, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

print(f'Training EEGConformerGAP for {conformer_cv_best_epoch} epochs...\n')
for epoch in range(1, conformer_cv_best_epoch + 1):
    loss, win_acc = run_epoch(all_loader, final_conformer, criterion, optimizer)
    print(f'  Epoch {epoch:02d}/{conformer_cv_best_epoch} | loss {loss:.4f} | window_acc {win_acc:.3f}')

torch.save(final_conformer.state_dict(), 'conformer_intra_final.pt')
print('\nSaved conformer_intra_final.pt')

## 8. Test Evaluation

Primary metric: **file accuracy** — window logits averaged per file, then argmax.
With 8 test files, accuracy moves in steps of 12.5%.

In [ ]:
def predict_logits_conformer(windows):
    final_conformer.eval()
    all_logits = []
    with torch.no_grad():
        for i in range(0, len(windows), 128):
            x = torch.from_numpy(windows[i:i+128]).unsqueeze(1).to(device)
            all_logits.append(final_conformer(x).cpu().numpy())
    return np.concatenate(all_logits, axis=0)


conf_metrics = summarize_logits(predict_logits_conformer(X_test), y_test, test_groups)
print(f'Window accuracy : {conf_metrics["window_acc"]:.3f}  (diagnostic only)')
print(f'File accuracy   : {conf_metrics["file_acc"]:.3f}  <- primary metric')
print(f'\nConfusion matrix (rows=true, cols=pred):')
print(f'Classes: {CLASS_NAMES}')
print(conf_metrics['file_confusion_matrix'])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
cm = conf_metrics['file_confusion_matrix']
im = axes[0].imshow(cm, cmap='Blues')
axes[0].set_xticks(range(4)); axes[0].set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
axes[0].set_yticks(range(4)); axes[0].set_yticklabels(CLASS_NAMES)
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
axes[0].set_title(f'EEGConformerGAP\nFile acc: {conf_metrics["file_acc"]:.3f}')
for i in range(4):
    for j in range(4):
        axes[0].text(j, i, cm[i, j], ha='center', va='center',
                     color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.colorbar(im, ax=axes[0])

axes[1].plot(_diag_history['train_loss'], label='Train loss', color='steelblue')
axes[1].plot(_diag_history['val_loss'],   label='Val loss',   color='darkorange')
axes[1].axvline(conformer_cv_best_epoch - 1, color='red', linestyle='--',
                linewidth=1.2, label=f'Epoch used ({conformer_cv_best_epoch})')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].set_title('Diagnostic loss curves'); axes[1].legend()
plt.tight_layout(); plt.show()